# Uplift Modeling for HCP Educational Outreach

This notebook demonstrates an end-to-end uplift modeling workflow using a fully synthetic randomized experiment.

**Decision question:** which HCPs are most likely to change behavior *because of* an educational outreach intervention?

## 1. Setup and synthetic data

The data-generating process includes both baseline response propensity and heterogeneous treatment effects.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))
import uplift_modeling as um

df = um.make_data()
df.head()


## 2. Randomization check

Treatment is assigned 50/50 by construction. Feature means should therefore be similar across treatment groups, up to sampling variation.

In [ ]:
df.groupby('treatment')[um.FEATURES].mean().round(3)


## 3. Train/test split and models

We compare a conventional response model against three treatment-effect estimators: T-learner, S-learner, and a doubly robust learner.

In [ ]:
train, test = train_test_split(
    df, test_size=0.35, random_state=um.SEED, stratify=df['treatment']
)
train, test = train.reset_index(drop=True), test.reset_index(drop=True)

test['response_score'] = um.fit_response_model(train, test)
test['uplift_t'], *_ = um.fit_t_learner(train, test)
test['uplift_s'] = um.fit_s_learner(train, test)
test['uplift_dr'] = um.fit_dr_learner(train, test)


## 4. Why response prediction is not enough

A response model ranks HCPs by expected outcome. Uplift models rank them by the expected *difference* between treatment and control outcomes.

In [ ]:
rows = []
for label, score in [
    ('Response model', 'response_score'),
    ('T-learner', 'uplift_t'),
    ('S-learner', 'uplift_s'),
    ('DR-learner', 'uplift_dr'),
]:
    rows.append({
        'model': label,
        'qini_style_area': um.qini_style_area(test, score),
        'corr_with_true_uplift': test[[score, 'true_uplift']].corr().iloc[0,1],
        'uplift_top_20pct': um.top_fraction_uplift(test, score, 0.20),
    })
pd.DataFrame(rows)


## 5. Uplift by decile with uncertainty

The top-ranked groups should, on average, exhibit larger treated-control differences. Perfect monotonicity is not expected because each decile is a finite randomized sample.

In [ ]:
deciles = um.uplift_by_decile(test, 'uplift_t')
deciles


## 6. Business interpretation

If outreach capacity is constrained, ranking by estimated uplift supports allocating the intervention to HCPs expected to generate the largest *incremental* response rather than simply the highest baseline response probability.

In a real deployment, the targeting policy should be prospectively validated and reviewed within the appropriate privacy, governance, medical, legal, and promotional-compliance framework.

## 7. Extension to observational data

This example uses randomized treatment. In observational settings, treatment assignment may be confounded. Extensions would include propensity modeling, overlap diagnostics, doubly robust estimation, causal forests, sensitivity analysis, and stronger identification assumptions.